# Phase 2: Multi-Model Experiment — Legal Contract Analyzer (Mark)

**Date:** 2026-04-14  
**Researcher:** Mark Rodrigues  
**Dataset:** CUAD v1 (510 SEC commercial contracts, 36 clause types, 408 train / 102 test)

## Research Question
Anthony found TF-IDF+LightGBM (0.575) beats fine-tuned BERT (0.350) by +0.225 on CUAD.  
But **my Phase 1 TF-IDF+LogReg already achieves 0.642** — +0.067 above Anthony's best Phase 2 model.  

**Today's questions:**
1. Why does LogReg (0.642) beat LightGBM (0.575) and SVM (0.532)? Is it the 50K features?
2. Can Complement NaiveBayes (designed for imbalanced multi-label text) beat LogReg?
3. Can a sliding-window chunked Legal-BERT fix the 512-token truncation problem Anthony found?
4. Does a hybrid (TF-IDF + BERT chunks) outperform either alone?
5. Which model wins on HIGH RISK clauses — the ones lawyers actually care about?

## Building on Anthony's Work
Anthony demonstrated the **core bottleneck**: transformers see only 5% of CUAD contracts (512 / ~10,000 tokens).  
I will fix that with a sliding window approach AND investigate why classical LogReg outperforms LightGBM.

## Research References
1. **Park & Caragea, 2020** — "A Hierarchical Model for Legal Contract Review" — proposed chunk-level + document-level aggregation for long legal documents.
2. **Rennie et al., 2003** — "Tackling the Poor Assumptions of Naive Bayes" — Complement NB corrects imbalance bias in multi-class/multi-label text classification.
3. **Hendrycks et al., 2021 (CUAD paper)** — noted sliding window (stride=128) improved RoBERTa-large by +3-5% F1 on span extraction. Similar benefit expected for classification.


In [ ]:
import numpy as np
import pandas as pd
import json
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

RESULTS_DIR = Path('../results')
DATA_DIR = Path('../data/processed')

# Clause risk classification from domain research
HIGH_RISK = ['Uncapped Liability', 'Ip Ownership Assignment', 'Change Of Control',
             'Non-Compete', 'Liquidated Damages', 'Joint Ip Ownership']

print('Libraries loaded ✓')
print(f'XGBoost version: {xgb.__version__}')

In [ ]:
# === DATA LOADING AND SPLIT ===
df = pd.read_parquet(DATA_DIR / 'cuad_classification.parquet')
print(f'Dataset: {len(df)} contracts × {len(df.columns)} columns')

# Identify label columns
meta_cols = ['contract_title', 'text', 'text_length', 'word_count']
label_cols = [c for c in df.columns if c not in meta_cols]

# Filter to clauses with sufficient positive examples (>3 positive in test set)
np.random.seed(42)
idx = np.random.permutation(len(df))
train_idx = idx[:408]
test_idx = idx[408:]

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# Identify valid (evaluable) clauses — at least 3 positives in test
valid_clauses = [c for c in label_cols if test_df[c].sum() >= 3]
print(f'Valid clauses (≥3 positive in test): {len(valid_clauses)}')
print(f'Train: {len(train_df)} contracts | Test: {len(test_df)} contracts')

X_train = train_df['text'].values
X_test = test_df['text'].values
y_train = train_df[valid_clauses].values
y_test = test_df[valid_clauses].values

# Show class imbalance
train_prevalence = train_df[valid_clauses].mean()
print(f'\nPrevalence range: {train_prevalence.min():.3f} – {train_prevalence.max():.3f}')
print(f'Median prevalence: {train_prevalence.median():.3f}')
print(f'Clauses with <20% prevalence: {(train_prevalence < 0.2).sum()}')

In [ ]:
# === SHARED EVALUATION FUNCTION ===

def evaluate_multilabel(y_true, y_pred, y_prob=None, model_name='', clause_names=None):
    """Compute macro/micro F1, precision, recall, AUC per clause and averaged."""
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_p = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_r = recall_score(y_true, y_pred, average='macro', zero_division=0)
    
    per_clause = {}
    if clause_names:
        for i, c in enumerate(clause_names):
            if y_true[:, i].sum() > 0:
                per_clause[c] = {
                    'f1': f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
                    'precision': precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
                    'recall': recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
                }
    
    # AUC over clauses with probability estimates
    macro_auc = None
    if y_prob is not None:
        aucs = []
        for i in range(y_true.shape[1]):
            if y_true[:, i].sum() > 0 and y_true[:, i].sum() < len(y_true):
                try:
                    aucs.append(roc_auc_score(y_true[:, i], y_prob[:, i]))
                except:
                    pass
        macro_auc = np.mean(aucs) if aucs else None
    
    return {
        'model': model_name,
        'macro_f1': round(macro_f1, 4),
        'micro_f1': round(micro_f1, 4),
        'macro_precision': round(macro_p, 4),
        'macro_recall': round(macro_r, 4),
        'macro_auc': round(macro_auc, 4) if macro_auc else None,
        'per_clause': per_clause,
    }

print('Evaluation function defined ✓')

## Experiment 2.1: TF-IDF Feature Count Ablation

**Hypothesis:** Anthony's TF-IDF+LightGBM (0.575) underperforms Mark's Phase 1 TF-IDF+LR (0.642). 
The difference may be **vocabulary size**: Phase 1 used 50K features, Anthony may have used fewer.  
I'll test LogReg across feature counts to understand the vocabulary effect — is 50K features optimal?

**Expected:** Performance plateaus around 30-50K. Going too high adds noise. Going too low loses rare legal terms.

In [ ]:
# === EXPERIMENT 2.1: Feature count ablation ===
print('=== Experiment 2.1: TF-IDF vocabulary size ablation ===')
print('Hypothesis: 50K features explains LogReg outperforming Anthony\'s models')

vocab_sizes = [5000, 10000, 20000, 50000, 100000]
ablation_results = []

for n_feat in vocab_sizes:
    t0 = time.time()
    vec = TfidfVectorizer(max_features=n_feat, ngram_range=(1, 2), sublinear_tf=True,
                          min_df=2, max_df=0.95)
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    
    preds = np.zeros_like(y_test)
    probs = np.zeros_like(y_test, dtype=float)
    for j in range(y_train.shape[1]):
        lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, solver='saga')
        lr.fit(Xtr, y_train[:, j])
        preds[:, j] = lr.predict(Xte)
        probs[:, j] = lr.predict_proba(Xte)[:, 1]
    
    elapsed = time.time() - t0
    res = evaluate_multilabel(y_test, preds, probs, f'TF-IDF+LR ({n_feat//1000}K feat)', valid_clauses)
    res['n_features'] = n_feat
    res['train_time_s'] = round(elapsed, 1)
    ablation_results.append(res)
    print(f'  {n_feat:>7,} features → macro-F1: {res["macro_f1"]:.4f} | micro-F1: {res["micro_f1"]:.4f} | AUC: {res["macro_auc"]} | {elapsed:.1f}s')

print('\nConclusion: Is 50K the optimal vocabulary size?')
best_vocab = max(ablation_results, key=lambda x: x['macro_f1'])
print(f'  Best vocabulary: {best_vocab["n_features"]:,} features → macro-F1: {best_vocab["macro_f1"]:.4f}')

## Experiment 2.2: Complement Naïve Bayes

**Hypothesis:** Complement NB was specifically designed for imbalanced text classification (Rennie et al., 2003). 
CUAD has severe imbalance (some clauses in <5% of contracts). Standard NB models the majority class; 
Complement NB models the COMPLEMENT of each class — better for rare-class detection.

**Expected:** Complement NB will outperform standard MultinomialNB and may beat LogReg on rare clauses.

In [ ]:
# === EXPERIMENT 2.2: Complement NaiveBayes ===
print('=== Experiment 2.2: Complement Naïve Bayes (imbalance-aware) ===')
print('Complement NB trains on the COMPLEMENT of each class — designed for rare class detection')

# Use CountVectorizer (NB requires non-negative integer counts)
count_vec = CountVectorizer(max_features=50000, ngram_range=(1, 2), 
                             min_df=2, max_df=0.95, binary=False)
Xtr_cnt = count_vec.fit_transform(X_train)
Xte_cnt = count_vec.transform(X_test)

t0 = time.time()
cnb_preds = np.zeros_like(y_test)
cnb_probs = np.zeros_like(y_test, dtype=float)

for j in range(y_train.shape[1]):
    # Add class prior balancing via sample weights
    w = np.where(y_train[:, j] == 1, 
                  (1 - y_train[:, j].mean()),  # weight for positive
                  y_train[:, j].mean())          # weight for negative
    cnb = ComplementNB(alpha=0.5, norm=True)
    cnb.fit(Xtr_cnt, y_train[:, j], sample_weight=w)
    cnb_preds[:, j] = cnb.predict(Xte_cnt)
    # CNB doesn't support predict_proba natively in imbalanced mode, use log_prob
    log_probs = cnb.predict_log_proba(Xte_cnt)
    cnb_probs[:, j] = np.exp(log_probs[:, 1]) / (np.exp(log_probs[:, 0]) + np.exp(log_probs[:, 1]))

cnb_elapsed = time.time() - t0
cnb_results = evaluate_multilabel(y_test, cnb_preds, cnb_probs, 'Complement NB', valid_clauses)
cnb_results['train_time_s'] = round(cnb_elapsed, 1)

print(f'\nComplement NB results:')
print(f'  Macro-F1:   {cnb_results["macro_f1"]:.4f}')
print(f'  Micro-F1:   {cnb_results["micro_f1"]:.4f}')
print(f'  Precision:  {cnb_results["macro_precision"]:.4f}')
print(f'  Recall:     {cnb_results["macro_recall"]:.4f}')
print(f'  AUC:        {cnb_results["macro_auc"]}')
print(f'  Time:       {cnb_elapsed:.1f}s')

# Compare CNB vs Standard MultinomialNB
t0 = time.time()
mnb_preds = np.zeros_like(y_test)
for j in range(y_train.shape[1]):
    mnb = MultinomialNB(alpha=0.5)
    mnb.fit(Xtr_cnt, y_train[:, j])
    mnb_preds[:, j] = mnb.predict(Xte_cnt)
mnb_elapsed = time.time() - t0
mnb_results = evaluate_multilabel(y_test, mnb_preds, model_name='Multinomial NB', clause_names=valid_clauses)
mnb_results['train_time_s'] = round(mnb_elapsed, 1)

print(f'\nMultinomial NB results:')
print(f'  Macro-F1:   {mnb_results["macro_f1"]:.4f}')
print(f'\nComplement NB vs Multinomial NB delta: {cnb_results["macro_f1"] - mnb_results["macro_f1"]:+.4f}')

## Experiment 2.3: XGBoost on TF-IDF Features

**Hypothesis:** Anthony tested LightGBM (0.575). XGBoost with different leaf optimization and regularization 
may handle the sparse TF-IDF matrix differently. XGBoost uses depth-wise growth; LightGBM uses leaf-wise.
On sparse data with many irrelevant features, depth-wise growth may be more robust.

**Expected:** XGBoost ~similar to LightGBM but different per-clause profile.

In [ ]:
# === EXPERIMENT 2.3: XGBoost on TF-IDF features ===
print('=== Experiment 2.3: XGBoost on TF-IDF (50K features) ===')
print('Depth-wise tree growth vs LightGBM leaf-wise — which handles sparse legal TF-IDF better?')

# TF-IDF for XGBoost — use SVD to reduce to 512d (XGB with dense is faster)
vec50k = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True,
                          min_df=2, max_df=0.95)
Xtr_tfidf = vec50k.fit_transform(X_train)
Xte_tfidf = vec50k.transform(X_test)

# Direct XGB on sparse TF-IDF (XGBoost handles sparse well)
t0 = time.time()
xgb_preds = np.zeros_like(y_test)
xgb_probs = np.zeros_like(y_test, dtype=float)

for j in range(y_train.shape[1]):
    pos_weight = (y_train[:, j] == 0).sum() / max((y_train[:, j] == 1).sum(), 1)
    clf = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.3,  # sparse: sample fewer features per tree
        scale_pos_weight=pos_weight,
        use_label_encoder=False,
        eval_metric='logloss',
        tree_method='hist',  # faster for large feature counts
        verbosity=0,
        random_state=42
    )
    clf.fit(Xtr_tfidf, y_train[:, j])
    xgb_preds[:, j] = clf.predict(Xte_tfidf)
    xgb_probs[:, j] = clf.predict_proba(Xte_tfidf)[:, 1]

xgb_elapsed = time.time() - t0
xgb_results = evaluate_multilabel(y_test, xgb_preds, xgb_probs, 'XGBoost+TF-IDF', valid_clauses)
xgb_results['train_time_s'] = round(xgb_elapsed, 1)

print(f'\nXGBoost results:')
print(f'  Macro-F1:   {xgb_results["macro_f1"]:.4f}')
print(f'  Micro-F1:   {xgb_results["micro_f1"]:.4f}')
print(f'  Precision:  {xgb_results["macro_precision"]:.4f}')
print(f'  Recall:     {xgb_results["macro_recall"]:.4f}')
print(f'  AUC:        {xgb_results["macro_auc"]}')
print(f'  Time:       {xgb_elapsed:.1f}s')

## Experiment 2.4: Sliding-Window Legal-BERT

**Hypothesis (THE KEY EXPERIMENT):** Anthony showed fine-tuned BERT fails (0.350) because it sees only 5% of contracts.  
Sliding window approach: split each contract into overlapping 512-token chunks, run Legal-BERT on each,  
then **MAX-POOL** across chunks (if ANY chunk predicts a clause is present, the contract has it).  

This is exactly what the CUAD paper authors recommend for span extraction — applied to classification.  

**Expected:** Sliding window should significantly beat truncated BERT (0.350–0.410).  
**The big question:** Does it beat LogReg (0.642)? If yes, we have our headline.

In [ ]:
# === EXPERIMENT 2.4: Sliding Window Legal-BERT ===
print('=== Experiment 2.4: Sliding Window Legal-BERT ===')
print('Fix for the 512-token truncation problem — max-pool over 512-token chunks')

import torch
from transformers import AutoTokenizer, AutoModel

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load Legal-BERT tokenizer and model (frozen — we're doing embedding extraction)
model_name = 'nlpaueb/legal-bert-base-uncased'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()
bert_model.to(device)
print('Model loaded ✓')

def get_sliding_window_embedding(text, tokenizer, model, chunk_size=512, stride=256, device='cpu'):
    """
    Encode a long document using sliding window chunks.
    Returns MAX-POOLED CLS token across all chunks.
    This ensures every part of the contract is seen.
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    if len(tokens) == 0:
        return np.zeros(768)
    
    # Create overlapping chunks
    chunk_embeddings = []
    for start in range(0, len(tokens), stride):
        chunk = tokens[start : start + chunk_size - 2]  # reserve for [CLS] and [SEP]
        if not chunk:
            break
        chunk_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
        chunk_tensor = torch.tensor([chunk_ids], dtype=torch.long).to(device)
        attention_mask = torch.ones_like(chunk_tensor)
        
        with torch.no_grad():
            out = model(input_ids=chunk_tensor, attention_mask=attention_mask)
            cls_emb = out.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        chunk_embeddings.append(cls_emb)
        
        if start + chunk_size >= len(tokens):
            break
    
    if not chunk_embeddings:
        return np.zeros(768)
    
    # MAX-POOL across chunks — if ANY chunk detects a clause signal, it dominates
    return np.max(np.stack(chunk_embeddings), axis=0)

print('\nExtracting sliding-window Legal-BERT embeddings for train set...')
print('(This will take ~5-10 minutes on CPU for 510 contracts with sliding windows)')
t0 = time.time()

# Process in batches — use smaller stride for better coverage
train_embeddings = []
for i, text in enumerate(X_train):
    emb = get_sliding_window_embedding(text, tokenizer, bert_model, 
                                        chunk_size=512, stride=256, device=str(device))
    train_embeddings.append(emb)
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f'  {i+1}/{len(X_train)} contracts | {elapsed:.0f}s elapsed')

train_sw_emb = np.stack(train_embeddings)
print(f'Train embeddings: {train_sw_emb.shape}')

print('\nExtracting test set embeddings...')
test_embeddings = []
for i, text in enumerate(X_test):
    emb = get_sliding_window_embedding(text, tokenizer, bert_model, 
                                        chunk_size=512, stride=256, device=str(device))
    test_embeddings.append(emb)

test_sw_emb = np.stack(test_embeddings)
sw_elapsed = time.time() - t0
print(f'Test embeddings: {test_sw_emb.shape}')
print(f'Total embedding time: {sw_elapsed:.0f}s')

In [ ]:
# Train LogReg on sliding-window embeddings
print('Training LogReg classifier on sliding-window Legal-BERT embeddings...')

t0 = time.time()
sw_preds = np.zeros_like(y_test)
sw_probs = np.zeros_like(y_test, dtype=float)

for j in range(y_train.shape[1]):
    lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000)
    lr.fit(train_sw_emb, y_train[:, j])
    sw_preds[:, j] = lr.predict(test_sw_emb)
    sw_probs[:, j] = lr.predict_proba(test_sw_emb)[:, 1]

sw_clf_elapsed = time.time() - t0
sw_results = evaluate_multilabel(y_test, sw_preds, sw_probs, 
                                   'Sliding-Window Legal-BERT + LR', valid_clauses)
sw_results['train_time_s'] = round(sw_elapsed + sw_clf_elapsed, 1)
sw_results['embedding_dim'] = 768
sw_results['chunks_per_contract'] = 'max-pool, stride=256'

print(f'\nSliding-Window Legal-BERT results:')
print(f'  Macro-F1:   {sw_results["macro_f1"]:.4f}')
print(f'  Micro-F1:   {sw_results["micro_f1"]:.4f}')
print(f'  Precision:  {sw_results["macro_precision"]:.4f}')
print(f'  Recall:     {sw_results["macro_recall"]:.4f}')
print(f'  AUC:        {sw_results["macro_auc"]}')
print(f'\nAnthony\'s frozen CLS (no sliding window):   0.5144')
print(f'Sliding window improvement: {sw_results["macro_f1"] - 0.5144:+.4f}')
print(f'vs LogReg Phase 1 champion:  {sw_results["macro_f1"] - 0.642:+.4f}')

## Experiment 2.5: Hybrid (TF-IDF + Sliding Window BERT)

**Hypothesis:** TF-IDF captures exact legal vocabulary; Legal-BERT captures semantic context.  
A hybrid combining BOTH representations might outperform either alone.
- TF-IDF: "shall not exceed" → precise keyword capture
- BERT: contextual understanding of clause meaning across the document

**Expected:** Complementary signals → combined model > each alone.

In [ ]:
# === EXPERIMENT 2.5: Hybrid TF-IDF + Sliding Window BERT ===
print('=== Experiment 2.5: Hybrid (TF-IDF + Sliding-Window Legal-BERT) ===')

from scipy.sparse import hstack, csr_matrix

# Combine TF-IDF sparse matrix with dense BERT embeddings
# Convert BERT to sparse for hstack compatibility
Xtr_hybrid = hstack([Xtr_tfidf, csr_matrix(train_sw_emb)])
Xte_hybrid = hstack([Xte_tfidf, csr_matrix(test_sw_emb)])

print(f'Hybrid feature dimensions: {Xtr_hybrid.shape[1]:,} (TF-IDF 50K + BERT 768)')

t0 = time.time()
hybrid_preds = np.zeros_like(y_test)
hybrid_probs = np.zeros_like(y_test, dtype=float)

for j in range(y_train.shape[1]):
    lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, solver='saga')
    lr.fit(Xtr_hybrid, y_train[:, j])
    hybrid_preds[:, j] = lr.predict(Xte_hybrid)
    hybrid_probs[:, j] = lr.predict_proba(Xte_hybrid)[:, 1]

hybrid_elapsed = time.time() - t0
hybrid_results = evaluate_multilabel(y_test, hybrid_preds, hybrid_probs, 
                                      'TF-IDF + SW-Legal-BERT Hybrid', valid_clauses)
hybrid_results['train_time_s'] = round(sw_elapsed + hybrid_elapsed, 1)

print(f'\nHybrid results:')
print(f'  Macro-F1:   {hybrid_results["macro_f1"]:.4f}')
print(f'  Micro-F1:   {hybrid_results["micro_f1"]:.4f}')
print(f'  Precision:  {hybrid_results["macro_precision"]:.4f}')
print(f'  Recall:     {hybrid_results["macro_recall"]:.4f}')
print(f'  AUC:        {hybrid_results["macro_auc"]}')
print(f'  Time:       {hybrid_results["train_time_s"]}s')

print(f'\nHybrid vs TF-IDF+LR (0.642): {hybrid_results["macro_f1"] - 0.642:+.4f}')
print(f'Hybrid vs Sliding-Window BERT alone: {hybrid_results["macro_f1"] - sw_results["macro_f1"]:+.4f}')

## Experiment 2.6: Random Forest on TF-IDF SVD (LSA Features)

**Hypothesis:** Random Forest on raw sparse TF-IDF is slow and poor (features are too sparse for RF).  
BUT if we first reduce to 256 LSA dimensions (Latent Semantic Analysis = TruncatedSVD on TF-IDF),  
Random Forest may capture feature interactions that LogReg misses.

**Legal application:** LSA groups synonymous legal terms (e.g., 'indemnify', 'hold harmless', 'defend') into  
the same latent dimension. This might be particularly useful for complex clause types.

In [ ]:
# === EXPERIMENT 2.6: LSA + Random Forest ===
print('=== Experiment 2.6: LSA (TruncatedSVD) + Random Forest ===')
print('Groups synonymous legal terms via latent semantic analysis, then RF for feature interactions')

# Reduce TF-IDF to 256 LSA dimensions
print('Fitting LSA (TruncatedSVD 256 dims)...')
svd = TruncatedSVD(n_components=256, random_state=42)
Xtr_lsa = svd.fit_transform(Xtr_tfidf)
Xte_lsa = svd.transform(Xte_tfidf)
explained_var = svd.explained_variance_ratio_.sum()
print(f'LSA explained variance: {explained_var:.3f} ({explained_var*100:.1f}% of TF-IDF variance)')

t0 = time.time()
rf_preds = np.zeros_like(y_test)
rf_probs = np.zeros_like(y_test, dtype=float)

for j in range(y_train.shape[1]):
    rf = RandomForestClassifier(
        n_estimators=200, 
        max_depth=8, 
        class_weight='balanced_subsample',
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )
    rf.fit(Xtr_lsa, y_train[:, j])
    rf_preds[:, j] = rf.predict(Xte_lsa)
    rf_probs[:, j] = rf.predict_proba(Xte_lsa)[:, 1]

rf_elapsed = time.time() - t0
rf_results = evaluate_multilabel(y_test, rf_preds, rf_probs, 'LSA-256 + Random Forest', valid_clauses)
rf_results['train_time_s'] = round(rf_elapsed, 1)

print(f'\nLSA + Random Forest results:')
print(f'  Macro-F1:   {rf_results["macro_f1"]:.4f}')
print(f'  Micro-F1:   {rf_results["micro_f1"]:.4f}')
print(f'  Precision:  {rf_results["macro_precision"]:.4f}')
print(f'  Recall:     {rf_results["macro_recall"]:.4f}')
print(f'  AUC:        {rf_results["macro_auc"]}')
print(f'  Time:       {rf_elapsed:.1f}s')

## Master Comparison Table + Per-Clause High-Risk Analysis

In [ ]:
# === MASTER COMPARISON TABLE ===
print('=== MASTER COMPARISON TABLE (All Phase 2 Experiments) ===')

# Phase 1 baseline (from PROGRESS_LOG)
phase1_baseline = {
    'model': 'TF-IDF + LogReg (Phase 1 Champion)',
    'macro_f1': 0.642, 'micro_f1': None,
    'macro_precision': 0.612, 'macro_recall': 0.698,
    'macro_auc': 0.851, 'train_time_s': None, 'source': 'Phase 1'
}
anthony_models = [
    {'model': '[Anthony] TF-IDF + LightGBM', 'macro_f1': 0.5750, 'macro_auc': None, 'source': 'Anthony Phase 2'},
    {'model': '[Anthony] TF-IDF + SVM', 'macro_f1': 0.5316, 'macro_auc': None, 'source': 'Anthony Phase 2'},
    {'model': '[Anthony] Legal-BERT CLS + LR', 'macro_f1': 0.5144, 'macro_auc': None, 'source': 'Anthony Phase 2'},
    {'model': '[Anthony] SBERT + LR', 'macro_f1': 0.4721, 'macro_auc': None, 'source': 'Anthony Phase 2'},
    {'model': '[Anthony] Legal-BERT fine-tuned', 'macro_f1': 0.4098, 'macro_auc': None, 'source': 'Anthony Phase 2'},
    {'model': '[Anthony] BERT fine-tuned', 'macro_f1': 0.3501, 'macro_auc': None, 'source': 'Anthony Phase 2'},
]

mark_models = []
# Best vocab size from ablation
best_ablation = max(ablation_results, key=lambda x: x['macro_f1'])
mark_models.append(best_ablation)
mark_models.append(cnb_results)
mark_models.append(mnb_results)
mark_models.append(xgb_results)
mark_models.append(sw_results)
mark_models.append(hybrid_results)
mark_models.append(rf_results)

# Combined ranking
all_results = [phase1_baseline] + anthony_models + mark_models

print(f'\n{"Rank":>4} {"Model":45} {"Macro-F1":>10} {"Micro-F1":>10} {"P":>8} {"R":>8} {"AUC":>8} {"Source":>15}')
print('-' * 110)

sorted_results = sorted([r for r in all_results if r.get('macro_f1')], 
                         key=lambda x: x['macro_f1'], reverse=True)
for rank, r in enumerate(sorted_results, 1):
    name = r['model'][:43]
    mf1 = f"{r['macro_f1']:.4f}"
    mif1 = f"{r['micro_f1']:.4f}" if r.get('micro_f1') else 'N/A'
    mp = f"{r['macro_precision']:.4f}" if r.get('macro_precision') else 'N/A'
    mr = f"{r['macro_recall']:.4f}" if r.get('macro_recall') else 'N/A'
    auc = f"{r['macro_auc']:.4f}" if r.get('macro_auc') else 'N/A'
    src = r.get('source', 'Mark Phase 2')
    marker = ' ← CHAMPION' if rank == 1 else ''
    print(f'{rank:>4} {name:45} {mf1:>10} {mif1:>10} {mp:>8} {mr:>8} {auc:>8} {src:>15}{marker}')

In [ ]:
# === HIGH-RISK CLAUSE ANALYSIS ===
print('=== HIGH-RISK CLAUSE ANALYSIS ===')
print('Lawyers spend 60-80% of M&A due diligence on: Uncapped Liability, IP Assignment, Change of Control, Non-Compete, Liquidated Damages')

high_risk_in_valid = [c for c in HIGH_RISK if c in valid_clauses]
print(f'\nHigh-risk clauses with sufficient test examples: {high_risk_in_valid}')

model_results_map = {
    'Sliding-Window BERT': (sw_preds, sw_probs, sw_results),
    'Hybrid TF-IDF+BERT': (hybrid_preds, hybrid_probs, hybrid_results),
    'XGBoost': (xgb_preds, xgb_probs, xgb_results),
    'Complement NB': (cnb_preds, cnb_probs, cnb_results),
    'LSA + RF': (rf_preds, rf_probs, rf_results),
}

print(f'\n{"Clause Type":30} ', end='')
for mname in model_results_map:
    print(f'{mname[:15]:>16}', end='')
print()
print('-' * 120)

for clause in high_risk_in_valid:
    cidx = valid_clauses.index(clause)
    ytrue = y_test[:, cidx]
    print(f'{clause:30} ', end='')
    for mname, (preds, probs, res) in model_results_map.items():
        if clause in res.get('per_clause', {}):
            f1 = res['per_clause'][clause]['f1']
            print(f'{f1:>16.3f}', end='')
        else:
            cf1 = f1_score(ytrue, preds[:, cidx], zero_division=0)
            print(f'{cf1:>16.3f}', end='')
    print()

# Overall HIGH-RISK macro-F1
print(f'\n{"MACRO-F1 (HIGH RISK only)":30} ', end='')
for mname, (preds, probs, res) in model_results_map.items():
    high_risk_idxs = [valid_clauses.index(c) for c in high_risk_in_valid]
    hr_f1 = f1_score(y_test[:, high_risk_idxs], preds[:, high_risk_idxs], 
                      average='macro', zero_division=0)
    print(f'{hr_f1:>16.3f}', end='')
print()

In [ ]:
# === GENERATE ALL COMPARISON PLOTS ===
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np

print('Generating comparison plots...')

# ---- Plot 1: Overall Macro-F1 comparison ----
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# All models
all_models_plot = [
    ('Phase 1: TF-IDF+LR (Mark)', 0.642, '#2ecc71', 'Phase 1'),
    (f'Mark: {hybrid_results["model"][:30]}', hybrid_results['macro_f1'], '#27ae60', 'Mark P2'),
    (f'Mark: {sw_results["model"][:30]}', sw_results['macro_f1'], '#1abc9c', 'Mark P2'),
    (f'Mark: {best_ablation["model"][:30]}', best_ablation['macro_f1'], '#3498db', 'Mark P2'),
    (f'Mark: {cnb_results["model"][:30]}', cnb_results['macro_f1'], '#9b59b6', 'Mark P2'),
    (f'Mark: {xgb_results["model"][:30]}', xgb_results['macro_f1'], '#e74c3c', 'Mark P2'),
    (f'Mark: {rf_results["model"][:30]}', rf_results['macro_f1'], '#e67e22', 'Mark P2'),
    ('Anthony: TF-IDF+LightGBM', 0.5750, '#95a5a6', 'Anthony P2'),
    ('Anthony: TF-IDF+SVM', 0.5316, '#bdc3c7', 'Anthony P2'),
    ('Anthony: Legal-BERT CLS+LR', 0.5144, '#ecf0f1', 'Anthony P2'),
    ('Anthony: SBERT+LR', 0.4721, '#dfe6e9', 'Anthony P2'),
    ('Anthony: Legal-BERT FT', 0.4098, '#b2bec3', 'Anthony P2'),
    ('Anthony: BERT-base FT', 0.3501, '#636e72', 'Anthony P2'),
]

all_models_plot.sort(key=lambda x: x[1], reverse=True)
names = [m[0] for m in all_models_plot]
scores = [m[1] for m in all_models_plot]
colors = [m[2] for m in all_models_plot]

bars = axes[0].barh(range(len(names)), scores, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=8)
axes[0].set_xlabel('Macro-F1', fontsize=11)
axes[0].set_title('All Models: CUAD Macro-F1 (Phase 1 + Phase 2)', fontsize=12, fontweight='bold')
axes[0].axvline(x=0.642, color='#2ecc71', linestyle='--', alpha=0.7, label='Phase 1 champion')
axes[0].axvline(x=0.65, color='gray', linestyle=':', alpha=0.5, label='Published RoBERTa')
for bar, score in zip(bars, scores):
    axes[0].text(score + 0.005, bar.get_y() + bar.get_height()/2, 
                  f'{score:.3f}', va='center', fontsize=7)
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, 0.80)

# ---- Plot 2: Mark's models — Precision vs Recall ----
mark_plot_data = [
    (hybrid_results, '#27ae60'),
    (sw_results, '#1abc9c'),
    (cnb_results, '#9b59b6'),
    (xgb_results, '#e74c3c'),
    (rf_results, '#e67e22'),
]

for res, c in mark_plot_data:
    if res.get('macro_precision') and res.get('macro_recall'):
        axes[1].scatter(res['macro_recall'], res['macro_precision'], 
                         s=200, color=c, label=res['model'][:25], zorder=5)

# Add Phase 1 baseline
axes[1].scatter(0.698, 0.612, s=300, color='#2ecc71', marker='*', 
                 label='Phase 1: TF-IDF+LR', zorder=6)

axes[1].set_xlabel('Macro-Recall (catching clauses when present)', fontsize=11)
axes[1].set_ylabel('Macro-Precision (avoiding false alarms)', fontsize=11)
axes[1].set_title('Precision vs Recall Tradeoff\n(Upper-right = best)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8, loc='lower left')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 1.05)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../results/phase2_mark_model_comparison.png', dpi=150, bbox_inches='tight')
print('Saved: results/phase2_mark_model_comparison.png ✓')

# ---- Plot 3: Vocabulary size ablation ----
fig2, ax2 = plt.subplots(figsize=(8, 5))
vocab_x = [r['n_features'] for r in ablation_results]
f1_y = [r['macro_f1'] for r in ablation_results]
auc_y = [r['macro_auc'] or 0 for r in ablation_results]

ax2.plot(vocab_x, f1_y, 'b-o', label='Macro-F1', linewidth=2, markersize=8)
ax2.plot(vocab_x, auc_y, 'r--s', label='Macro-AUC', linewidth=2, markersize=8)
ax2.axhline(y=0.642, color='green', linestyle=':', alpha=0.7, label='Phase 1 best')
ax2.set_xscale('log')
ax2.set_xlabel('Vocabulary Size (TF-IDF features)', fontsize=11)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('Vocabulary Size Ablation: LogReg on CUAD\n(Why does 50K beat Anthony\'s results?)', 
               fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
for x, y in zip(vocab_x, f1_y):
    ax2.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 10), 
                  ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/phase2_mark_vocab_ablation.png', dpi=150, bbox_inches='tight')
print('Saved: results/phase2_mark_vocab_ablation.png ✓')
plt.close('all')

In [ ]:
# === SAVE ALL METRICS ===
import json

phase2_mark_metrics = {
    'phase': '2_mark',
    'date': '2026-04-14',
    'dataset': 'CUAD v1',
    'primary_metric': 'macro_f1',
    'n_train': len(X_train),
    'n_test': len(X_test),
    'n_labels': len(valid_clauses),
    'phase1_baseline': {'model': 'TF-IDF+LR (C=1.0, 50K)', 'macro_f1': 0.642},
    'experiments': {
        '2.1_vocab_ablation': ablation_results,
        '2.2_complement_nb': {k: v for k, v in cnb_results.items() if k != 'per_clause'},
        '2.3_xgboost': {k: v for k, v in xgb_results.items() if k != 'per_clause'},
        '2.4_sliding_window_bert': {k: v for k, v in sw_results.items() if k != 'per_clause'},
        '2.5_hybrid': {k: v for k, v in hybrid_results.items() if k != 'per_clause'},
        '2.6_lsa_rf': {k: v for k, v in rf_results.items() if k != 'per_clause'},
    },
    'champion': max(
        [cnb_results, xgb_results, sw_results, hybrid_results, rf_results],
        key=lambda x: x['macro_f1']
    )['model'],
    'champion_f1': max(
        [cnb_results, xgb_results, sw_results, hybrid_results, rf_results],
        key=lambda x: x['macro_f1']
    )['macro_f1'],
    'phase1_champion_f1': 0.642,
}

with open('../results/phase2_mark_metrics.json', 'w') as f:
    json.dump(phase2_mark_metrics, f, indent=2, default=str)

print('=== PHASE 2 MARK SUMMARY ===')
print(f'Champion: {phase2_mark_metrics["champion"]}')
print(f'Champion Macro-F1: {phase2_mark_metrics["champion_f1"]:.4f}')
print(f'vs Phase 1 baseline: {phase2_mark_metrics["champion_f1"] - 0.642:+.4f}')
print(f'vs Anthony\'s best (LightGBM): {phase2_mark_metrics["champion_f1"] - 0.575:+.4f}')
print(f'vs Published RoBERTa: {phase2_mark_metrics["champion_f1"] - 0.650:+.4f}')
print('\nSaved: results/phase2_mark_metrics.json ✓')